# Notebook 05 — Data Agent: Staff Persona

Configure and test an AI Data Agent for university staff. Staff have full access to all student, academic, and financial data.

Each demo question is paired with a reference SQL query to verify the agent's output.

## Data Agent System Prompt

When configuring the Data Agent in Fabric, use the following system prompt:

```
You are an AI assistant for a Singapore university. You help academic staff analyse
student data including enrolments, exam results, and financial transactions.
The data covers academic years 2021-2024.

Use the university schema with tables: dim_student, dim_course, dim_program,
dim_department, dim_staff, dim_academic_period, dim_exam_type, dim_fee_type,
dim_date, bridge_course_program, fact_enrollments, fact_exam_results,
fact_financial_transactions.

Currency is SGD. Grading follows the NUS 5.0 GPA scale (A+/A = 5.0, A- = 4.5,
B+ = 4.0, B = 3.5, B- = 3.0, C+ = 2.5, C = 2.0, D+ = 1.5, D = 1.0, F = 0.0).
Each module is worth 4 Modular Credits (MC). Pass grades are D and above (>= 40%).
```

## Q1: How many students are currently enrolled in each program?

**Expected insight:** Active student count per program, showing distribution across faculties.

In [ ]:
spark.sql("""
    SELECT p.program_name, p.program_type, COUNT(*) AS active_students
    FROM university.dim_student s
    JOIN university.dim_program p ON s.program_key = p.program_key
    WHERE s.enrolment_status = 'Active'
    GROUP BY p.program_name, p.program_type
    ORDER BY active_students DESC
""").show(20, truncate=False)

## Q2: What is the average GPA across all departments for the current academic year?

**Expected insight:** Department-level GPA comparison for 2024.

In [ ]:
spark.sql("""
    SELECT d.department_name,
           ROUND(AVG(er.grade_points), 2) AS avg_gpa,
           ROUND(AVG(er.score_percentage), 2) AS avg_score,
           COUNT(*) AS exam_count
    FROM university.fact_exam_results er
    JOIN university.dim_course c ON er.course_key = c.course_key
    JOIN university.dim_department d ON c.department_key = d.department_key
    JOIN university.dim_academic_period ap ON er.academic_period_key = ap.academic_period_key
    WHERE ap.academic_year = 2024
    GROUP BY d.department_name
    ORDER BY avg_gpa DESC
""").show(20, truncate=False)

## Q3: Which courses have the highest failure rates?

**Expected insight:** Courses where students struggle most, based on F grades in fact_exam_results.

In [ ]:
spark.sql("""
    SELECT c.course_id, c.course_name, d.department_name,
           COUNT(*) AS total_results,
           SUM(CASE WHEN er.grade_letter = 'F' THEN 1 ELSE 0 END) AS fail_count,
           ROUND(SUM(CASE WHEN er.grade_letter = 'F' THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 1) AS fail_rate_pct
    FROM university.fact_exam_results er
    JOIN university.dim_course c ON er.course_key = c.course_key
    JOIN university.dim_department d ON c.department_key = d.department_key
    GROUP BY c.course_id, c.course_name, d.department_name
    HAVING COUNT(*) >= 50
    ORDER BY fail_rate_pct DESC
    LIMIT 10
""").show(truncate=False)

## Q4: Show me the revenue breakdown by fee type for Semester 1 2024

**Expected insight:** Tuition dominates, with smaller contributions from misc fees, accommodation, and fines.

In [ ]:
spark.sql("""
    SELECT ft_type.fee_type_name, ft_type.fee_category,
           ROUND(SUM(ft.amount), 2) AS total_amount_sgd,
           COUNT(*) AS transaction_count
    FROM university.fact_financial_transactions ft
    JOIN university.dim_fee_type ft_type ON ft.fee_type_key = ft_type.fee_type_key
    JOIN university.dim_academic_period ap ON ft.academic_period_key = ap.academic_period_key
    WHERE ap.academic_year = 2024 AND ap.semester = 'Semester 1'
      AND ft.transaction_type = 'Charge'
    GROUP BY ft_type.fee_type_name, ft_type.fee_category
    ORDER BY total_amount_sgd DESC
""").show(truncate=False)

## Q5: Compare domestic vs international student performance in Computer Science courses

**Expected insight:** Score and GPA comparison between the two groups in CS courses.

In [ ]:
spark.sql("""
    SELECT s.domestic_international,
           ROUND(AVG(er.score_percentage), 2) AS avg_score,
           ROUND(AVG(er.grade_points), 2) AS avg_gpa,
           COUNT(*) AS exam_count
    FROM university.fact_exam_results er
    JOIN university.dim_student s ON er.student_key = s.student_key
    JOIN university.dim_course c ON er.course_key = c.course_key
    JOIN university.dim_department d ON c.department_key = d.department_key
    WHERE d.department_name = 'Computer Science'
    GROUP BY s.domestic_international
""").show(truncate=False)

## Q6: Which students are at risk of academic probation (GPA below 2.0)?

**Expected insight:** Students whose average grade_points across all exams is below 2.0.

In [ ]:
spark.sql("""
    SELECT s.student_id, s.first_name, s.last_name,
           p.program_name, s.enrolment_status,
           ROUND(AVG(er.grade_points), 2) AS avg_gpa,
           COUNT(*) AS total_exams
    FROM university.fact_exam_results er
    JOIN university.dim_student s ON er.student_key = s.student_key
    JOIN university.dim_program p ON s.program_key = p.program_key
    WHERE s.enrolment_status = 'Active'
    GROUP BY s.student_id, s.first_name, s.last_name, p.program_name, s.enrolment_status
    HAVING AVG(er.grade_points) < 2.0
    ORDER BY avg_gpa ASC
""").show(20, truncate=False)

## Q7: What is the semester-over-semester enrolment trend for the Business School?

**Expected insight:** Enrolment counts per semester for Business Administration and Accountancy programs.

In [ ]:
spark.sql("""
    SELECT ap.academic_year, ap.semester, p.program_name,
           COUNT(*) AS enrolment_count
    FROM university.fact_enrollments fe
    JOIN university.dim_program p ON fe.program_key = p.program_key
    JOIN university.dim_department d ON p.department_key = d.department_key
    JOIN university.dim_academic_period ap ON fe.academic_period_key = ap.academic_period_key
    WHERE d.faculty = 'Business School'
    GROUP BY ap.academic_year, ap.semester, p.program_name
    ORDER BY ap.academic_year, ap.semester, p.program_name
""").show(50, truncate=False)

## Q8: How much scholarship funding was disbursed per program in the last 2 years?

**Expected insight:** Scholarship credit totals (SGD) by program for 2023-2024.

In [ ]:
spark.sql("""
    SELECT p.program_name,
           ROUND(SUM(ft.amount), 2) AS total_scholarship_sgd,
           COUNT(DISTINCT ft.student_key) AS scholarship_students
    FROM university.fact_financial_transactions ft
    JOIN university.dim_student s ON ft.student_key = s.student_key
    JOIN university.dim_program p ON s.program_key = p.program_key
    JOIN university.dim_academic_period ap ON ft.academic_period_key = ap.academic_period_key
    WHERE ft.transaction_type = 'Credit'
      AND ap.academic_year >= 2023
    GROUP BY p.program_name
    ORDER BY total_scholarship_sgd DESC
""").show(20, truncate=False)

## Q9: Which staff members coordinate the most courses?

**Expected insight:** Top course coordinators by course count, with department info.

In [ ]:
spark.sql("""
    SELECT st.staff_id, st.first_name, st.last_name,
           st.role_title, d.department_name,
           COUNT(*) AS courses_coordinated
    FROM university.dim_course c
    JOIN university.dim_staff st ON c.coordinator_staff_key = st.staff_key
    JOIN university.dim_department d ON st.department_key = d.department_key
    GROUP BY st.staff_id, st.first_name, st.last_name, st.role_title, d.department_name
    ORDER BY courses_coordinated DESC
    LIMIT 10
""").show(truncate=False)

## Q10: What percentage of students complete their program within the expected duration?

**Expected insight:** On-time graduation rate, comparing graduated students' actual vs expected timeline.

In [ ]:
spark.sql("""
    SELECT p.program_name,
           COUNT(*) AS graduated_count,
           SUM(CASE WHEN s.expected_graduation_date >= CURRENT_DATE() THEN 1 ELSE 0 END) AS on_track,
           ROUND(SUM(CASE WHEN s.expected_graduation_date >= CURRENT_DATE() THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 1) AS on_track_pct
    FROM university.dim_student s
    JOIN university.dim_program p ON s.program_key = p.program_key
    WHERE s.enrolment_status = 'Graduated'
    GROUP BY p.program_name
    ORDER BY on_track_pct DESC
""").show(20, truncate=False)

## Staff Persona Summary

The staff Data Agent can answer questions across all dimensions:
- **Enrolment analytics** — trends, program distribution, completion rates
- **Academic performance** — GPA by department, failure rates, at-risk students
- **Financial data** — revenue, payments, scholarships, overdue amounts
- **Operational data** — staff workload, course coordination

**Next Steps:** Proceed to **Notebook 06** for the student persona with RLS.